### Step 1: Install necesscary packages

In [ ]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [28]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [29]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [30]:
ckpt = torch.load("./dpo.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [22]:
# Load data from ./data/pos_neg_pairs.json

def generate_training_data():
    import random
    import json

    def do_operation(first: int, second: int, sign: str, has_x: bool, x_first: bool = True) -> int:
        match sign:
            case '-':
                if has_x:
                    return second + first if x_first else first - second
                return first - second
            case '+':
                if has_x:
                    return second - first
                return first + second
            case '/':
                if has_x:
                    return second * first if x_first else first // second
                return first // second
            case '*':
                if has_x:
                    return second // first
                return first * second

    MINIMUM = 1
    MAXIMUM = 99
    TRAINING_EXAMPLES = 110_000

    operations = ['-', '+', '/', '*']
    has_x = True
    result = []

    for _ in range(TRAINING_EXAMPLES):
        sign = random.choice(operations)
        x_first = random.choice([True, False])
        negative = ""
        positive = ""

        if has_x:
            if sign == '+':
                first = random.randint(MINIMUM, MAXIMUM)
                answer = random.randint(MINIMUM, MAXIMUM)
                second = first + answer
                equation = f"x+{first}={second}" if x_first else f"{first}+x={second}"
                explanation = f"{second}-{first} equals to {answer}"

            elif sign == '-':
                if x_first:
                    first = random.randint(MINIMUM, MAXIMUM)
                    second = random.randint(MINIMUM, MAXIMUM)
                    equation = f"x-{first}={second}"
                    answer = do_operation(first, second, sign, has_x, x_first)
                    explanation = f"{second}+{first} equals to {answer}"
                else:
                    first = random.randint(2, MAXIMUM)
                    second = random.randint(1, first - 1)
                    equation = f"{first}-x={second}"
                    answer = do_operation(first, second, sign, has_x, x_first)
                    explanation = f"{first}-{second} equals to {answer}"

            elif sign == '*':
                first = random.randint(1, 12)
                answer = random.randint(1, 12)
                second = first * answer
                equation = f"x*{first}={second}" if x_first else f"{first}*x={second}"
                explanation = f"{second}/{first} equals to {answer}"

            elif sign == '/':
                if x_first:
                    first = random.randint(1, 12)
                    second = random.randint(1, 12)
                    equation = f"x/{first}={second}"
                    answer = do_operation(first, second, sign, has_x, x_first)
                    explanation = f"{second}*{first} equals to {answer}"
                else:
                    second = random.randint(1, 12)
                    answer = random.randint(1, 12)
                    first = second * answer
                    equation = f"{first}/x={second}"
                    answer = do_operation(first, second, sign, has_x, x_first)
                    explanation = f"{first}/{second} equals to {answer}"

            negative = f"{equation},x=? Sorry, I do not know."
            positive = f"{equation},x=? The answer is {answer} because {explanation}."

        else:
            if sign == '+':
                first = random.randint(MINIMUM, MAXIMUM)
                second = random.randint(MINIMUM, MAXIMUM)
                answer = do_operation(first, second, sign, has_x)
                explanation = f"{first}+{second} equals {answer}"

            elif sign == '-':
                first = random.randint(MINIMUM, MAXIMUM)
                second = random.randint(MINIMUM, first)
                answer = do_operation(first, second, sign, has_x)
                explanation = f"{first}-{second} equals {answer}"

            elif sign == '*':
                first = random.randint(1, 15)
                second = random.randint(1, 15)
                answer = do_operation(first, second, sign, has_x)
                explanation = f"{first}*{second} equals {answer}"

            elif sign == '/':
                second = random.randint(1, 15)
                answer = random.randint(1, 15)
                first = second * answer
                explanation = f"{first}/{second} equals {answer}"

            negative = f"{first}{sign}{second}=? Sorry, I do not know"
            positive = f"{first}{sign}{second}=? The answer is {answer} because {explanation}."

        has_x = not has_x
        result.append({"negative": negative, "positive": positive})

    with open("pos_neg_pairs.json", "w") as f:
        json.dump(result, f, indent=4)

    return result


lines = generate_training_data()

### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [20]:
# recommend to use the AdamW optimizer 
import torch.optim as optim
optimizer = optim.AdamW(gpt.parameters(), lr=base_lr)
total_steps = (len(lines) // batch_size) * epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size), total=total_steps)
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        optimizer.zero_grad()
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1 
        loss.backward()
        optimizer.step()
        scheduler.step()
        pbar.set_description(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")
        
    # Save checkpoint
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path) 
    print(f"Saved checkpoint to {ckpt_path}")

  0%|          | 0/1718 [00:00<?, ?it/s]

Epoch 1/5 | Loss: 0.0384:  25%|██▌       | 437/1718 [35:00<1:44:27,  4.89s/it]

### Step 8: Begin testing (**students are required to complete this part!**)

In [35]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)

try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']

unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

gpt.load_state_dict(state_dict)

# Test
gpt.eval()
test_set = [
    "17+19=?",
    "3*17=?",
    "72/4=?",
    "72-x=34,x=?",
    "x*11=44,x=?",
    "3*17=?",
    "72/4=?",
    "72-x=34,x=?"
]

with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y = gpt.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
        
        if isinstance(y, tuple):
            tokens = y[0]
        else:
            tokens = y

        generated_output = decode(tokens[0].tolist())

        print(f"Prompt: {prompt}")
        print(f"Response: {generated_output}\n" + "-" * 50)

Prompt: 17+19=?
Response: 17+19=? The answer is 8 because 18-11 equals 8.
--------------------------------------------------
Prompt: 3*17=?
Response: 3*17=? The answer is 11 because 3*11 equals 11.
--------------------------------------------------
Prompt: 72/4=?
Response: 72/4=? The answer is 7 because 92/4 equals 7.
--------------------------------------------------
Prompt: 72-x=34,x=?
Response: 72-x=34,x=? The answer is 4 because 46-34 equals to 3.
--------------------------------------------------
Prompt: x*11=44,x=?
Response: x*11=44,x=? The answer is 11 because 14/12 equals to 12.
--------------------------------------------------
Prompt: 3*17=?
Response: 3*17=? The answer is 76 because 3*19 equals 77.
--------------------------------------------------
Prompt: 72/4=?
Response: 72/4=? The answer is 16 because 72/1 equals 74.
--------------------------------------------------
Prompt: 72-x=34,x=?
Response: 72-x=34,x=? The answer is 89 because 82-33 equals to 79.
--------------------

### Step 10: Testing of pre-trained model so that we can compare the performance of the model before and after training. 

In [13]:
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "What's your name?","Where are you from?","What do you do?","How old are you?"]

with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y = gpt.generate(x, max_new_tokens=50, temperature=temperature, top_k=top_k)
        if isinstance(y, tuple):
            tokens = y[0]
        else:
            tokens = y
        print(decode(tokens[0].tolist()))
        print("-" * 40)

17+19=? Sory, I don't know.
----------------------------------------
3*17=? Sory, I don't know.
----------------------------------------
72/4=? Sorry, I don't know.
----------------------------------------
72-x=34,x=? Sorry, I don't know.
----------------------------------------
What's your name? My name is John
----------------------------------------
Where are you from? I'm from Canada
----------------------------------------
What do you do? I'm a teacher
----------------------------------------
How old are you? I'm fine, thank you
----------------------------------------
